# P2_81 — M3 strict dual-blocked generator × prompt sensitivity

**Status: NOT YET EXECUTED.** Run this notebook on a Colab GPU and send back the final ZIP.

Purpose: test whether ModernBERT Condition-C recovery survives a stricter split in which every test row belongs to both an unseen `Prompt_ID` fold and an unseen anonymous generator stratum. Generator identity is used only as a blocking variable; checkpoint names are never supplied to the classifier.

For each target, outer prompt fold `f`, and anonymous generator stratum `g`:
- test = rows with `Fold=f` **and** `Generator_Stratum=g`;
- training = rows with `Fold!=f` **and** `Generator_Stratum!=g`;
- an inner development fold is selected only from the remaining prompt folds and also excludes generator `g`;
- epoch count is selected inside this restricted training population;
- the model is reinitialised, refit on the full restricted outer-training population, and evaluated once on the test cell.

This yields one OOF prediction for every case while blocking both prompt and generator overlap.

In [ ]:
from pathlib import Path
import os, sys, shutil, gc, json, hashlib, zipfile, platform

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

PROJECT_DIR = Path('/content/drive/MyDrive/P2') if IN_COLAB else Path('/content/P2')
INPUT_DIR = PROJECT_DIR/'input'
RESULTS_DIR = PROJECT_DIR/'results'
OUT_DIR = RESULTS_DIR/'P2_81_M3_DUAL_BLOCKED'
LOCAL_CACHE = Path('/content/p281_m3_hf_cache')
LOCAL_TMP = Path('/content/p281_m3_tmp')
for d in [INPUT_DIR,RESULTS_DIR,OUT_DIR,LOCAL_CACHE,LOCAL_TMP]: d.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME']=str(LOCAL_CACHE)
os.environ['TRANSFORMERS_CACHE']=str(LOCAL_CACHE/'transformers')
os.environ['TMPDIR']=str(LOCAL_TMP)
print('Output:', OUT_DIR)

In [ ]:
%pip -q install -U "transformers>=4.57,<5" datasets accelerate safetensors pandas openpyxl "scikit-learn>=1.4"

In [ ]:
import numpy as np, pandas as pd, torch, transformers
from collections import Counter
from datasets import Dataset
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
    Trainer, TrainingArguments, EarlyStoppingCallback, set_seed
)
assert torch.cuda.is_available(), 'Select a GPU runtime.'
print('GPU:', torch.cuda.get_device_name(0))
print('transformers:', transformers.__version__)

In [ ]:
GOLD_NAME='P2_FINAL_SENIOR_ADJUDICATED_GOLD_v1.0.xlsx'
BASE_NAME='P2_CPU_Baselines_v1.0.xlsx'
EXPECTED_SHA256={
 GOLD_NAME:'ccc910b7bafbfa9607e93ef2eba8605f56f7ed87460ef74892d4877c7db4de65',
 BASE_NAME:'4502e71bd6d1b6d2941b0b10650caaa433d7188bf958017d2fc646d328a31163',
}
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for ch in iter(lambda:f.read(1024*1024),b''): h.update(ch)
    return h.hexdigest()
for n,e in EXPECTED_SHA256.items():
    p=INPUT_DIR/n
    assert p.exists(), f'Missing {p}'
    o=sha256(p); print(n,o); assert o==e,'Frozen input hash mismatch.'

gold=pd.read_excel(INPUT_DIR/GOLD_NAME,sheet_name='FINAL_GOLD')
oof={t:pd.read_excel(INPUT_DIR/BASE_NAME,sheet_name=f'OOF_{t}') for t in ['T1','T2','T3','T4']}
assert len(gold)==720 and gold.Case_ID.nunique()==720 and gold.Prompt_ID.nunique()==120
# Anonymous generator blocking variable: stable within-Prompt_ID row order 0..5.
assert gold.groupby('Prompt_ID',sort=False).size().eq(6).all()
gold=gold.copy().reset_index(drop=True)
gold['Generator_Stratum']=gold.groupby('Prompt_ID',sort=False).cumcount().astype(int)
gold_by_case=gold.set_index('Case_ID',drop=False)
for t in oof:
    oof[t]=oof[t].merge(gold[['Case_ID','Generator_Stratum']],on='Case_ID',validate='one_to_one')
print({t:len(d) for t,d in oof.items()})

In [ ]:
SEED=20260908
M3_MODEL_ID='answerdotai/ModernBERT-base'
M3_MODEL_REVISION='c0e44438c79d5a72972fe8b14dbbc822418356c9'
MAX_LENGTH=1024
LEARNING_RATE=2e-5
WEIGHT_DECAY=.01
MAX_SELECT_EPOCHS=8
TRAIN_BATCH=8
EVAL_BATCH=16
GRAD_ACCUM=2
EARLY_STOPPING_PATIENCE=2
BOOT_REPS=2000
TARGETS_TO_RUN=['T1','T2','T3','T4']  # You may run T3/T4 first, but final report should use all four.

def serialise_C(g):
    prompt=str(g['Prompt_Text'] if pd.notna(g['Prompt_Text']) else '').strip()
    source=str(g['Source_Context'] if pd.notna(g['Source_Context']) else '').strip()
    output=str(g['Model_Output'] if pd.notna(g['Model_Output']) else '').strip()
    if str(g['Source_Available']).lower()=='yes' and source:
        return f'Task:\n{prompt}\n\nAuthorised source/context:\n{source}\n\nResponse:\n{output}'
    return f'Task:\n{prompt}\n\nResponse:\n{output}'

tokenizer=AutoTokenizer.from_pretrained(M3_MODEL_ID,revision=M3_MODEL_REVISION)


In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self,*args,class_weights=None,**kwargs):
        super().__init__(*args,**kwargs); self.class_weights=class_weights
    def compute_loss(self,model,inputs,return_outputs=False,num_items_in_batch=None):
        labels=inputs.pop('labels'); out=model(**inputs)
        loss=torch.nn.functional.cross_entropy(out.logits,labels,weight=self.class_weights.to(out.logits.device))
        return (loss,out) if return_outputs else loss

def compute_metrics_eval(eval_pred):
    logits,labels=eval_pred
    return {'macro_f1':f1_score(labels,np.argmax(logits,axis=-1),average='macro')}

def make_dataset(rows,label_to_idx):
    return Dataset.from_dict({
        'text':[serialise_C(gold_by_case.loc[str(cid)]) for cid in rows.Case_ID],
        'labels':[label_to_idx[int(v)] for v in rows.y_true],
        'case_id':[str(x) for x in rows.Case_ID],
    })

def tokenise(ds):
    ids=list(ds['case_id'])
    def fn(batch): return tokenizer(batch['text'],truncation=True,max_length=MAX_LENGTH)
    return ds.map(fn,batched=True,remove_columns=['text','case_id']),ids

def weights_for(rows,label_to_idx):
    c=Counter(label_to_idx[int(v)] for v in rows.y_true); n=sum(c.values()); k=len(label_to_idx)
    assert set(c)==set(range(k)),f'Missing training class: {c}'
    return torch.tensor([n/(k*c[i]) for i in range(k)],dtype=torch.float32)

def selected_epoch(trainer):
    cand=[(float(x['eval_macro_f1']),float(x['epoch'])) for x in trainer.state.log_history if 'eval_macro_f1' in x and 'epoch' in x]
    assert cand
    best=max(v for v,_ in cand); epochs=[e for v,e in cand if abs(v-best)<1e-12]
    return max(1,int(round(min(epochs)))),best

In [ ]:
def train_cell(target,outer_fold,heldout_generator):
    rows=oof[target].copy()
    labels=sorted(rows.y_true.astype(int).unique())
    label_to_idx={v:i for i,v in enumerate(labels)}; idx_to_label={i:v for v,i in label_to_idx.items()}
    test=rows[(rows.Fold.astype(int)==outer_fold)&(rows.Generator_Stratum.astype(int)==heldout_generator)].copy()
    outer_train=rows[(rows.Fold.astype(int)!=outer_fold)&(rows.Generator_Stratum.astype(int)!=heldout_generator)].copy()
    assert len(test)>0
    assert set(outer_train.y_true.astype(int).unique())==set(labels)

    available=sorted(outer_train.Fold.astype(int).unique())
    preferred=(outer_fold+1)%5
    dev_fold=preferred if preferred in available else available[0]
    inner_dev=outer_train[outer_train.Fold.astype(int)==dev_fold].copy()
    inner_train=outer_train[outer_train.Fold.astype(int)!=dev_fold].copy()
    assert set(inner_train.y_true.astype(int).unique())==set(labels)

    # Stage 1: epoch selection inside the doubly restricted population.
    tr,_=tokenise(make_dataset(inner_train,label_to_idx)); dv,_=tokenise(make_dataset(inner_dev,label_to_idx))
    select_dir=LOCAL_TMP/f'select_{target}_f{outer_fold}_g{heldout_generator}'
    shutil.rmtree(select_dir,ignore_errors=True); select_dir.mkdir(parents=True,exist_ok=True)
    set_seed(SEED)
    m=AutoModelForSequenceClassification.from_pretrained(M3_MODEL_ID,revision=M3_MODEL_REVISION,num_labels=len(labels))
    bf=torch.cuda.is_bf16_supported()
    args=TrainingArguments(
        output_dir=str(select_dir),learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,
        num_train_epochs=MAX_SELECT_EPOCHS,per_device_train_batch_size=TRAIN_BATCH,
        per_device_eval_batch_size=EVAL_BATCH,gradient_accumulation_steps=GRAD_ACCUM,
        eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,
        metric_for_best_model='macro_f1',greater_is_better=True,save_total_limit=1,
        bf16=bf,fp16=not bf,report_to='none',seed=SEED,data_seed=SEED,logging_steps=20,
    )
    trainer=WeightedTrainer(model=m,args=args,train_dataset=tr,eval_dataset=dv,
        data_collator=DataCollatorWithPadding(tokenizer),compute_metrics=compute_metrics_eval,
        class_weights=weights_for(inner_train,label_to_idx),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)])
    trainer.train(); epochs,best_dev=selected_epoch(trainer)
    del trainer,m,tr,dv; gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(select_dir,ignore_errors=True)

    # Stage 2: fresh refit on all restricted outer-training rows, then one test pass.
    tr,_=tokenise(make_dataset(outer_train,label_to_idx)); te,_=tokenise(make_dataset(test,label_to_idx))
    refit_dir=LOCAL_TMP/f'refit_{target}_f{outer_fold}_g{heldout_generator}'
    shutil.rmtree(refit_dir,ignore_errors=True); refit_dir.mkdir(parents=True,exist_ok=True)
    set_seed(SEED)
    m=AutoModelForSequenceClassification.from_pretrained(M3_MODEL_ID,revision=M3_MODEL_REVISION,num_labels=len(labels))
    args=TrainingArguments(
        output_dir=str(refit_dir),learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,
        num_train_epochs=epochs,per_device_train_batch_size=TRAIN_BATCH,
        per_device_eval_batch_size=EVAL_BATCH,gradient_accumulation_steps=GRAD_ACCUM,
        eval_strategy='no',save_strategy='no',bf16=bf,fp16=not bf,report_to='none',
        seed=SEED,data_seed=SEED,logging_steps=20,
    )
    trainer=WeightedTrainer(model=m,args=args,train_dataset=tr,
        data_collator=DataCollatorWithPadding(tokenizer),class_weights=weights_for(outer_train,label_to_idx))
    trainer.train(); pr=trainer.predict(te)
    probs=torch.softmax(torch.tensor(pr.predictions),dim=-1).numpy(); ii=np.argmax(probs,axis=1)
    out=test[['Case_ID','Prompt_ID','Task_Type','Fold','Generator_Stratum','y_true','Human_Easy']].copy()
    out['M3_dual_pred']=[idx_to_label[int(i)] for i in ii]
    out['M3_dual_probs']=[json.dumps([float(x) for x in p]) for p in probs]
    out['selected_epochs']=epochs; out['inner_dev_fold']=dev_fold; out['best_inner_dev_macro_f1']=best_dev
    del trainer,m,tr,te; gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(refit_dir,ignore_errors=True)
    return out

In [ ]:
def run_target(target):
    parts=[]
    for f in range(5):
        for g in range(6):
            p=OUT_DIR/f'PART_{target}_f{f}_g{g}.csv'
            if p.exists():
                x=pd.read_csv(p); print('reuse',p.name)
            else:
                print('TRAIN',target,'fold',f,'generator',g)
                x=train_cell(target,f,g); x.to_csv(p,index=False)
            parts.append(x)
    full=pd.concat(parts,ignore_index=True)
    expected=set(oof[target].Case_ID.astype(str))
    assert len(full)==len(expected) and full.Case_ID.astype(str).nunique()==len(expected)
    assert set(full.Case_ID.astype(str))==expected
    full.to_csv(OUT_DIR/f'OOF_M3_DUAL_BLOCKED_{target}_C.csv',index=False)
    return full

dual={}
for target in TARGETS_TO_RUN:
    dual[target]=run_target(target)

In [ ]:
def macro_from_cm(cm):
    tp=np.diag(cm).astype(float); fp=cm.sum(0)-tp; fn=cm.sum(1)-tp; den=2*tp+fp+fn
    return float(np.divide(2*tp,den,out=np.zeros_like(tp),where=den!=0).mean())
def group_cms(df,pred_col,labels):
    ix={v:i for i,v in enumerate(labels)}; arr=[]
    for _,s in df.assign(_pid=df.Prompt_ID.astype(str)).groupby('_pid',sort=True):
        cm=np.zeros((len(labels),len(labels)),int)
        for y,p in zip(s.y_true.astype(int),s[pred_col].astype(int)): cm[ix[y],ix[p]]+=1
        arr.append(cm)
    return np.stack(arr)
def delta_ci(df,a,b,labels,reps=BOOT_REPS,seed=SEED):
    ca,cb=group_cms(df,a,labels),group_cms(df,b,labels); rng=np.random.default_rng(seed)
    vals=[]
    for _ in range(reps):
        q=rng.choice(len(ca),size=len(ca),replace=True)
        vals.append(macro_from_cm(cb[q].sum(0))-macro_from_cm(ca[q].sum(0)))
    return np.percentile(vals,[2.5,97.5])

summary=[]
for target in TARGETS_TO_RUN:
    d=dual[target].copy()
    ref_path=RESULTS_DIR/f'OOF_M3_{target}_C_seed20260908.csv'
    assert ref_path.exists(),f'Missing original M3-C OOF: {ref_path}'
    ref=pd.read_csv(ref_path)[['Case_ID','M3_pred']].rename(columns={'M3_pred':'M3_standard_pred'})
    d=d.merge(ref,on='Case_ID',validate='one_to_one')
    labels=sorted(d.y_true.astype(int).unique())
    std=f1_score(d.y_true,d.M3_standard_pred,average='macro')
    new=f1_score(d.y_true,d.M3_dual_pred,average='macro')
    lo,hi=delta_ci(d,'M3_standard_pred','M3_dual_pred',labels)
    summary.append({'target':target,'n':len(d),'standard_M3_C_macro_f1':std,
                    'dual_blocked_M3_C_macro_f1':new,'delta':new-std,'ci_low':lo,'ci_high':hi})
summary=pd.DataFrame(summary)
display(summary)
summary.to_csv(OUT_DIR/'P2_81_M3_DUAL_BLOCKED_SUMMARY.csv',index=False)

In [ ]:
manifest={
 'protocol':'P2_81_M3_STRICT_DUAL_BLOCKED_v1.0','seed':SEED,
 'model_id':M3_MODEL_ID,'model_revision':M3_MODEL_REVISION,'condition':'C',
 'split':'test requires unseen Prompt_ID fold AND unseen anonymous generator stratum',
 'epoch_selection':'nested inside doubly restricted training population','bootstrap_reps':BOOT_REPS,
 'gold_sha256':sha256(INPUT_DIR/GOLD_NAME),'base_sha256':sha256(INPUT_DIR/BASE_NAME),
 'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'transformers':transformers.__version__,
}
(OUT_DIR/'P2_81_M3_DUAL_BLOCKED_MANIFEST.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
zip_path=PROJECT_DIR/'P2_81_M3_DUAL_BLOCKED_RESULTS.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob('*'):
        if p.is_file(): z.write(p,arcname=p.relative_to(OUT_DIR))
print('SEND THIS FILE BACK:',zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))